# comparing automatic and manual transcriptions of participant audio

In [129]:
import numpy as np
import pandas as pd
import quail
import pickle
import random
import re
import os
from nltk.corpus import stopwords

### choose 5 participants at random to manually transcribe

In [2]:
# load in participants dict
with open('../../data/pickles/id_maps.p', 'rb') as file:
    id_maps = pickle.load(file)

# random5 = random.sample(id_maps.keys(), 5)
# print(random5)


# Random 5 chosen:
# ['MD-013119-A-01', 'MD-102218-A-07', 'MD-020119-A-04', 'MD-102218-B-01', 'MD-020119-B-03']


{'MD-013119-A-01': {'session 1': 'debughKp9W/debug1QFCO',
  'session 2': 'debugSeRNG/debug5pTo9'},
 'MD-013119-A-02': {'session 1': 'debug5ytRS:debug22nvn',
  'session 2': 'debug3Z74G:debug6CQ6D'},
 'MD-013119-B-01': {'session 1': 'debuguhHBa:debug1HOCZ',
  'session 2': 'debug0GSk9:debugttOEY'},
 'MD-020119-A-02': {'session 1': 'debugfGFHT:debugLCOgS',
  'session 2': 'debug9ngWi:debugpMjGl'},
 'MD-020119-A-03': {'session 1': 'debugRQQWb:debugwDi2H',
  'session 2': 'debugWRfIG:debugJWnAi'},
 'MD-020119-A-04': {'session 1': 'debugo1MpX/debugfpXYH',
  'session 2': 'debugeyj0U/debugsaXvf'},
 'MD-020119-B-02': {'session 1': 'debugAU1cu:debugzdGmZ',
  'session 2': 'debug27k0b:debugOGsyi'},
 'MD-020119-B-03': {'session 1': 'debugxQ7lq/debugS1UNb',
  'session 2': 'debug41SJm/debugJW4AU'},
 'MD-020719-B-01': {'session 1': 'debugwHDQh:debugDtNml', 'session 2': None},
 'MD-020819-B-01': {'session 1': 'debug3RFCM:debuggg1Gs', 'session 2': None},
 'MD-101218-A-01': {'session 1': 'debugIEH2T:debugDL

In [153]:
random5 = ['MD-013119-A-01', 'MD-102218-A-07', 'MD-020119-A-04', 'MD-102218-B-01', 'MD-020119-B-03']

# get ses1 & ses1 psiturk ids
rand5_turkids = {k: id_maps[k] for k in random5}

### load in manual transcriptions

## automatic transcription

### create speech context from union of words in annotations

In [86]:
atlep1 = pd.read_pickle('../../data/annotations_dfs/atlep1.p')
atlep2 = pd.read_pickle('../../data/annotations_dfs/atlep2.p')
arrdev = pd.read_pickle('../../data/annotations_dfs/arrdev.p')

In [89]:
def meets_word_criteria(string):
    """
    Removes words with characters not wanted in auto transcriber speech context
    """
    
    good_word = True
    
    # remove words containing digits
    if any(char.isdigit() for char in string):
        good_word = False
    
    # remove words surrounded by single quotes and possessives (avoid duplicates in nested quotations & possessives)
    if string.startswith("'") or string.endswith("'") or string.endswith("'s"):
        good_word = False
        
    # remove unhelpful simple words
    if len(string) <= 2:
        good_word = False
    
    return good_word

In [123]:
def create_speech_context(df):
    """
    Creates episode-specific speech context from video annotations
    """
    
    # use Narrative details (internal and external), Characters on screen, Speech, Character speaking, and Setting
    word_cols = [df.columns[i] for i in [2,3,4,6,7,9]]
    
    # create single string of all text
    allwords = ' '.join(df.loc[:,word_cols].apply(lambda x: ' '.join(x.dropna()), axis=1).values.tolist())
    
    # remove all characters except spaces (catches \n and \t), letters, apostrophes, dashes
    no_punctuation = re.sub("[^\w\s'-]+", '', allwords)
    
    speech_context = []
    
    # split words into list
    for word in no_punctuation.split():
        # identify unique words that meet criteria
        if word.lower() not in speech_context and meets_word_criteria(word):
            speech_context.append(word.lower())

    # remove English stopwords
    speech_context_nostop = [word for word in speech_context if word not in stopwords.words('english')]
    
    return speech_context_nostop

In [124]:
atlep1_speech_context = create_speech_context(atlep1)
atlep2_speech_context = create_speech_context(atlep2)
arrdev_speech_context = create_speech_context(arrdev)

In [178]:
for sub, data in rand5_turkids.items():
    for ses, turkid in data.items():
        rand5_turkids[sub][ses] = turkid.replace('/',':')

In [179]:
rand5_turkids

{'MD-013119-A-01': {'session 1': 'debughKp9W:debug1QFCO',
  'session 2': 'debugSeRNG:debug5pTo9'},
 'MD-020119-A-04': {'session 1': 'debugo1MpX:debugfpXYH',
  'session 2': 'debugeyj0U:debugsaXvf'},
 'MD-020119-B-03': {'session 1': 'debugxQ7lq:debugS1UNb',
  'session 2': 'debug41SJm:debugJW4AU'},
 'MD-102218-A-07': {'session 1': 'debugWvhTI:debug65Thd',
  'session 2': 'debugTkyN6:debugdsr6W'},
 'MD-102218-B-01': {'session 1': 'debugqWZVC:debugZSccI',
  'session 2': 'debugRvgf2:debugpnybj'}}

In [177]:
# audiodir = os.path.abspath('../../data/audio/')
# keypath = os.path.abspath('../../../google-credentials/cloud-speech-credentials.json')

# for sid, data in rand5_turkids.items():
#     print(sid + ':')
#     for ses, turkid in data.items():
        
#         # get corrrect audio file names and speech context by session and condition
#         if ses == 'session 1':
#             audiofiles = [turkid+'-recall.wav', turkid+'-prediction.wav']
#             episode_context = atlep1_speech_context
#         elif ses == 'session 2':
#             audiofiles = [turkid+'-delayed.wav', turkid+'-recall.wav']
#             if 'A' in sid:
#                 episode_context = atlep2_speech_context
#             else:
#                 episode_context = arrdev_speech_context
        
#         for audiofile in audiofiles:
            
#             # SKIP ALREADY FINISHED ONE
#             if audiofile != 'debughKp9W:debug1QFCO-recall.wav':
            
#                 # find file in correct testroom dir
#                 path = os.path.join(audiodir,'room1',turkid, audiofile)
#                 if not os.path.isfile(path):
#                     path = os.path.join(audiodir,'room2',turkid, audiofile)
#                     if not os.path.isfile(path):
#                         print('AUDIO FILE NOT FOUND: ' + path)
#                         break

#                 # and decode the audio
#                 print('\tdecoding ' + ses + ': ' + audiofile + ' ...')
#                 quail.decode_speech(path=path, keypath=keypath, save=True, speech_context=episode_context,
#                                     enable_word_time_offsets=False)


MD-013119-A-01:
	decoding session 1: debughKp9W:debug1QFCO-prediction.wav ...
Decoding file 1 of 1
Audio clip is longer than 1 minute.  Splitting into 6 one minute segments...
Transcript: sell in the next episode there's a lot of like unresolved especially with the media rate we're going to see
Confidence: 0.8629385828971863
Transcript:  possibly in jail
Confidence: 0.9582188129425049
Transcript:  cuz they're suspects will probably need to be interrogated this whole situation might even depending on how things go
Confidence: 0.936951756477356
Transcript:  there's also like the unresolved relationship with his wife and tell her that you loved her but
Confidence: 0.9105949401855469
Transcript:  she
Confidence: 0.8663959503173828
Transcript: able to because he was
Confidence: 0.9060564041137695
Transcript:  probably not talk that out and she is probably very upset with the fact that like you could go to jail and like their daughter
Confidence: 0.883453369140625
Transcript:  his parents ar

Transcript: okay so the episode starts in there at the jail
Confidence: 0.9067544937133789
Transcript:  waiting the processing room for jail and they're both sitting
Confidence: 0.8865995407104492
Transcript:  relaxing music
Confidence: 0.6635990142822266
Transcript:  and they're kind of like joking around
Confidence: 0.7960208058357239
Transcript:  they're not too stressed but I'm in Earnest is a little star first time
Confidence: 0.7555550336837769
Transcript:  finger waves might not very stressed out at all
Confidence: 0.765211284160614
Transcript:  laugh at this one girl who's like
Confidence: 0.8423495292663574
Transcript: music video of but like advocating that she was a person
Confidence: 0.8422468304634094
Transcript:  paper boy
Confidence: 0.8152186870574951
Transcript:  gets called chili the processing window area has Bill already been in the system so you just like collected his stuff was about to leave but he asked about his cousin in the end
Confidence: 0.900809109210968
T

Transcript: so the episode starts like at night time and it's outside world looks like American guys confronting these other dude I guess what is his girlfriend and their an argument over him paying the other guy back for something they fight like a fairly arguing and then the guy Alfred Binet later thought I'd gone at this other guy and then the other like I got a gun to basically and he was not out and then Earnest marks trust called Alfred down but he works on his gone at all berries just there looking at the dog being like I seen that dog before the other guy
Confidence: 0.933836042881012
Transcript: artists like what is your friend on an aerial view really different neighborhoods and stuff and like a show is Atlanta like the title screen happen then it shows like a bunch of different houses like big ones like broken once it starts with I
Confidence: 0.9388049840927124
Transcript:  earn like listening to music and Bella consumes in his face then I like start zooming out this girl a

Transcript: so I think for the next episode it'll continue off of the present have I don't know if they'll do the whole time skipping again cuz I don't know if it's a normal thing for the TV show but I'm sure it'll be done with Brandon Alfred in jail or like them saying that they weren't the ones who shot them cuz I'm not sure who is the injured one also I'm sure it is in jail
Confidence: 0.9567316770553589
Transcript:  and learn to be really mad at Alfred and I think it will put a chink in the relationship in the beginning but then the resolve it by the end cuz I think some other stuff will happen I don't know what happened about Harvard is going to take a lot of this will be
Confidence: 0.8887574076652527
Transcript:  friend of kick-starting
Confidence: 0.959403395652771
Transcript:  calm fighter like the plot of this I think this might have been the first episode
Confidence: 0.9298027157783508
Transcript: basically I think they're going to be in jail I think I can have some talking 

Transcript: show starts off with Alfred Vernon jail actually like they're cracking jokes
Confidence: 0.9403348565101624
Transcript:  and
Confidence: 0.8393973112106323
Transcript:  asking earn why you didn't hide the fucking weed cuz he was like it's not it's
Confidence: 0.8995718955993652
Transcript:  starts knocking everything get arrested for weed and then that guy is okay and not the time or place but yet probably and then they're sitting there when you get bailed out of star when he was like oh my God that's ex-girlfriend like is T-Pain music video Turn back now and say like offered to drive across this girl's name and she looks kind of like look over to call their name and their your book backed up cuz it
Confidence: 0.9203258752822876
Transcript: and
Confidence: 0.8256008625030518
Transcript:  then I'll forget spell. By Darius and earn kick a bill. Your cuz he's not in the system is a bond hasn't been posted to earn stays in jail when offered a leaving tries to ask if you can ge

...

Transcript: in the next episode will because we know that
Confidence: 0.9485421180725098
Transcript:  mind
Confidence: 0.8342475891113281
Transcript:  Albert and Cassidy me
Confidence: 0.7105950117111206
Transcript:  assume that something to happen with that we don't know who actually
Confidence: 0.8610304594039917
Transcript:  shot the other guy cuz they all had guns in the scene next to the bar but clearly because aren't and I'll are in custody that one of the other guys was shot and presumably killed so we'll probably try to get out of a situation he's going to reach out to his parents for help especially with that but I don't think
Confidence: 0.9388927817344666
Transcript: then we'll be super interested in
Confidence: 0.9091478586196899
Transcript:  helping him because when she saw on TV that he was in custody she didn't seem worried or concerned she just thought sugar hadn't said those idiots so she'll probably want to protect Laudy and not get involved with
Confidence: 0.9444519

Transcript: the episode starts by introducing all the characters that are Audible
Confidence: 0.7554978728294373
Transcript:  torch the Dad's party to celebrate his retirement and we're introduced to George Michael Bluth Michael Buble's it's a blue Stanley who is the Son and he has been working at his dad's house development company B is expecting a promotion then we are introduced the mom and they're also introduced to Lindsey who is my sister
Confidence: 0.9266676902770996
Transcript: aspiring magician and then
Confidence: 0.8276661038398743
Transcript:  the other side you had a job since grad school student is trying to bunch of different things and then we see Lindsay's husband to she married an act of defiance his name is Tobias and then George Michael is Michael son he was a banana stand and
Confidence: 0.9188418984413147
Transcript:  and then it flashes to Michael and George Michael
Confidence: 0.9427257776260376
Transcript:  who lives in the Attic of the loose company's model h

Transcript: first episode opened your car door smoking a car guy in the driver's seat was wearing a black T-shirt with a gold chain around his neck it was an older car there outside of some storage at night somebody walked by the car and a broken mirror off the front of it
Confidence: 0.9446629285812378
Transcript: behind the driver seat was really upset by this got out of the car confronted the do you want the money to repay for the mirror
Confidence: 0.9138009548187256
Transcript:  when the guy wouldn't give him the money for the gun
Confidence: 0.9455651640892029
Transcript:  guy who's driving car the black t-shirt for the gun on the other guy there's a girl there also with the man who broke the mirror off a car
Confidence: 0.8876524567604065
Transcript: Kailua sitting in the driver's seat called the girl a bitch that made her angry made him more angry when she returned a verbal altercation
Confidence: 0.9151996970176697
Transcript:  then once the driver pulled a gun on the guy who 

Transcript: they're definitely a playoff what happens with a shooting I think got orange girlfriend or whatever to be pissed if he got arrested or as part of the shooting a sad thing I think paperboys The Paperboy
Confidence: 0.9108806252479553
Transcript:  I think this is going to screw up your future more than he hope so
Confidence: 0.8933869004249573
Transcript:  asking Jimmy Return of the man on the in the brown suit it was on the bus with Darius that night you had this like weird for you also a dog when he gets off and then goes into Boards of Poland orange life
Confidence: 0.8967265486717224
Transcript:  but I think Ernst girlfriend has to be really angry after the next episode possibly.
Confidence: 0.8305694460868835
Transcript: I think that's going to work chest a relationship in somewhere
Confidence: 0.687854528427124
Transcript:  magic herbs pair to get arrested to his relationship with play Boys music
Confidence: 0.8639467358589172
Transcript:  rat is The Paperboy
Confidence

Transcript: began with his family
Confidence: 0.933923065662384
Transcript:  boat
Confidence: 0.8920096158981323
Transcript:  retirement party for their dad or three brothers a daughter the Long Island Iced Tea Company party
Confidence: 0.8110610246658325
Transcript:  george-michael like main guy's name for a lot of names
Confidence: 0.8942777514457703
Transcript:  dads like announcing that he's giving his company away
Confidence: 0.873660683631897
Transcript:  dad has started the company from like a little banana stand
Confidence: 0.8643816113471985
Transcript:  turn on the boat and the s s s c u s s e comes up like a magic box
Confidence: 0.5062634348869324
Transcript: sister is on the boat with a pirate really protesting the yacht club for the exclusionary gay people accidentally ended up on the boat because he left the hotel Justin thought I was a Pirate Fairy like pirate themed parties we left the hotel beforehand with all the Pirates game ended up being a
Confidence: 0.9416847229

In [171]:
with open('../../data/audio/room1/debughKp9W:debug1QFCO/debughKp9W:debug1QFCO-recall.wav.p', 'rb') as f:
    test = pickle.load(f)

In [172]:
test

[results {
   alternatives {
     transcript: "okay so everything started and their two men African-American men and a car and are always an older model car"
     confidence: 0.8841516375541687
   }
 }
 results {
   alternatives {
     transcript: " pay slip out and chase after another guy who you later found out angrily broke one of his rear view mirrors"
     confidence: 0.8529588580131531
   }
 }
 results {
   alternatives {
     transcript: " answer ASAP about chase after the other guy obviously mad and yelling that I\'m one of them is his rapper name is Paperboy is yelling anyone\'s money to stick his rear view mirror"
     confidence: 0.9062669277191162
   }
 }
 results {
   alternatives {
     transcript: " and so the situation escalates because the person who broke"
     confidence: 0.9342687726020813
   }
 }, results {
   alternatives {
     transcript: "parent refuses to give them money"
     confidence: 0.9200791120529175
   }
 }
 results {
   alternatives {
     transcript:

In [175]:
quail.__file__

'/Users/paxtonfitzpatrick/Documents/Dartmouth/CDL/quail/quail/__init__.py'